**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (SysCall)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "SysCall"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

In [ ]:
data.head()

,Package_Name,Total_System_Calls,Unique_System_Calls,Unique_System_Calls_List,File_Operations,Unique_File_Operations,Unique_File_Operations_List,Memory_Operations,Unique_Memory_Operations,Unique_Memory_Operations_List,...,Filesystem_Operations,Unique_Filesystem_Operations,Unique_Filesystem_Operations_List,Security_Operations,Unique_Security_Operations,Unique_Security_Operations_List,Miscellaneous_Operations,Unique_Miscellaneous_Operations,Unique_Miscellaneous_Operations_List,Level
0,10Cent10-999.0.4.tar.gz,4859,33,"munmap, newfstatat, openat, fstat, ioctl, lsee...",3945,12,"newfstatat, mkdir, fstat, write, unlink, read,...",433,4,"munmap, mmap, mprotect, brk",...,0,0,NaN,0,0,NaN,7,2,"getrandom, uname",1
1,10Cent11-999.0.4.tar.gz,1423,34,"read, getpid, ioctl, write, wait4, close, newf...",1014,8,"newfstatat, write, fstat, read, rmdir, lseek, ...",25,4,"munmap, brk, mprotect, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1
2,11Cent-999.0.0.tar.gz,9966,42,"newfstatat, mkdir, openat, fstat, write, close...",8556,12,"newfstatat, mkdir, fstat, write, unlink, read,...",204,5,"mremap, munmap, brk, mmap, mprotect",...,0,0,NaN,0,0,NaN,8,2,"getrandom, uname",1
3,11Cent-999.0.1.tar.gz,1049,33,"restart_syscall, getsockopt, newfstatat, opena...",782,8,"newfstatat, openat, fstat, write, read, rmdir,...",11,3,"munmap, brk, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1
4,11Cent-999.0.2.tar.gz,791,33,"restart_syscall, read, brk, poll, uname, newfs...",625,9,"newfstatat, fstat, write, unlink, read, rmdir,...",5,3,"munmap, brk, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1


In [ ]:
if 'Package_Name' in data.columns:
    data = data.drop(columns=['Package_Name'])

# Combine categorical columns into a single text feature
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col != 'Level']
data['Combined_Categorical'] = data[categorical_columns].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the combined categorical text column using n-grams
vectorizer = CountVectorizer(ngram_range=(2, 3))
categorical_ngrams = vectorizer.fit_transform(data['Combined_Categorical'])

# Ensure only valid numeric columns are selected
numerical_columns = [
    col for col in data.columns
    if pd.to_numeric(data[col], errors='coerce').notnull().all() and col != 'Level'
]

# Prepare numerical features
numerical_features = data[numerical_columns].fillna(0).astype(float)

# Convert numerical features to sparse matrix
numerical_features_sparse = csr_matrix(numerical_features.values)

# Combine numerical features with n-gram features
X = hstack([numerical_features_sparse, categorical_ngrams])

# Scale features before combining
scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
X_scaled = scaler.fit_transform(X)

In [ ]:
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 08-29 21:03:59] {1680} INFO - task = classification
[flaml.automl.logger: 08-29 21:03:59] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 08-29 21:03:59] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 08-29 21:03:59] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 08-29 21:03:59] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-29 21:04:00] {2346} INFO - Estimated sufficient time budget=4420s. Estimated necessary time budget=102s.
[flaml.automl.logger: 08-29 21:04:00] {2398} INFO -  at 0.6s,	estimator lgbm's best error=0.0059,	best estimator lgbm's best error=0.0059
[flaml.automl.logger: 08-29 21:04:00] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-29 21:04:00] {2398} INFO -  at 1.0s,	estimator lgbm's best error=0.0051,	best estimator lgbm's best error=0.0051
[flaml.automl.logger: 08-29 21:04

[flaml.automl.logger: 08-29 21:04:26] {2398} INFO -  at 26.9s,	estimator extra_tree's best error=0.0043,	best estimator xgboost's best error=0.0018
[flaml.automl.logger: 08-29 21:04:26] {2219} INFO - iteration 34, current learner rf
[flaml.automl.logger: 08-29 21:04:28] {2398} INFO -  at 28.7s,	estimator rf's best error=0.0022,	best estimator xgboost's best error=0.0018
[flaml.automl.logger: 08-29 21:04:28] {2219} INFO - iteration 35, current learner xgboost
[flaml.automl.logger: 08-29 21:04:28] {2398} INFO -  at 28.9s,	estimator xgboost's best error=0.0018,	best estimator xgboost's best error=0.0018
[flaml.automl.logger: 08-29 21:04:28] {2219} INFO - iteration 36, current learner extra_tree
[flaml.automl.logger: 08-29 21:04:29] {2398} INFO -  at 30.2s,	estimator extra_tree's best error=0.0043,	best estimator xgboost's best error=0.0018
[flaml.automl.logger: 08-29 21:04:29] {2219} INFO - iteration 37, current learner xgboost
[flaml.automl.logger: 08-29 21:04:29] {2398} INFO -  at 30.4s

[flaml.automl.logger: 08-29 21:04:58] {2398} INFO -  at 59.3s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:04:58] {2219} INFO - iteration 70, current learner rf
[flaml.automl.logger: 08-29 21:05:00] {2398} INFO -  at 61.1s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:00] {2219} INFO - iteration 71, current learner rf
[flaml.automl.logger: 08-29 21:05:02] {2398} INFO -  at 63.1s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:02] {2219} INFO - iteration 72, current learner xgboost
[flaml.automl.logger: 08-29 21:05:02] {2398} INFO -  at 63.2s,	estimator xgboost's best error=0.0018,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:02] {2219} INFO - iteration 73, current learner rf
[flaml.automl.logger: 08-29 21:05:04] {2398} INFO -  at 64.7s,	estimator rf's best error=0.0017,	best estimato

[flaml.automl.logger: 08-29 21:05:26] {2219} INFO - iteration 106, current learner xgboost
[flaml.automl.logger: 08-29 21:05:26] {2398} INFO -  at 86.8s,	estimator xgboost's best error=0.0018,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:26] {2219} INFO - iteration 107, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:05:26] {2398} INFO -  at 87.1s,	estimator xgb_limitdepth's best error=0.0018,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:26] {2219} INFO - iteration 108, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:05:26] {2398} INFO -  at 87.3s,	estimator xgb_limitdepth's best error=0.0018,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:26] {2219} INFO - iteration 109, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:05:27] {2398} INFO -  at 87.6s,	estimator xgb_limitdepth's best error=0.0018,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:05:

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


...

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:06:50] {2398} INFO -  at 171.5s,	estimator lrl1's best error=0.3910,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:06:50] {2219} INFO - iteration 154, current learner rf


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:06:53] {2398} INFO -  at 174.0s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:06:53] {2219} INFO - iteration 155, current learner rf
[flaml.automl.logger: 08-29 21:06:55] {2398} INFO -  at 175.9s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:06:55] {2219} INFO - iteration 156, current learner rf
[flaml.automl.logger: 08-29 21:06:56] {2398} INFO -  at 177.5s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:06:56] {2219} INFO - iteration 157, current learner rf
[flaml.automl.logger: 08-29 21:06:59] {2398} INFO -  at 179.7s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:06:59] {2219} INFO - iteration 158, current learner rf
[flaml.automl.logger: 08-29 21:07:01] {2398} INFO -  at 182.2s,	estimator rf's best error=0.0017,	best estimator

[flaml.automl.logger: 08-29 21:08:01] {2398} INFO -  at 242.5s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:08:01] {2219} INFO - iteration 192, current learner rf
[flaml.automl.logger: 08-29 21:08:04] {2398} INFO -  at 244.7s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:08:04] {2219} INFO - iteration 193, current learner rf
[flaml.automl.logger: 08-29 21:08:05] {2398} INFO -  at 246.4s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:08:05] {2219} INFO - iteration 194, current learner rf
[flaml.automl.logger: 08-29 21:08:07] {2398} INFO -  at 248.2s,	estimator rf's best error=0.0017,	best estimator rf's best error=0.0017
[flaml.automl.logger: 08-29 21:08:07] {2219} INFO - iteration 195, current learner rf
[flaml.automl.logger: 08-29 21:08:10] {2398} INFO -  at 250.6s,	estimator rf's best error=0.0017,	best estimator

In [ ]:
selected_features = ['Unique_System_Calls', 'Unique_System_Calls_List', 'File_Operations', 'Unique_File_Operations_List', 'IO_Operations']